[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_02/11_repaso_prueba_2.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 11 — Repaso integrador de la Unidad 2

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 2**

Este notebook junta lo de las semanas 7 a 10 en dos problemas: una onda que
se propaga en un medio con pérdidas, y una interfaz vista en el ángulo de
Brewster. Úselo para comprobar sus desarrollos a mano.

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Evaluar un campo en un punto dado a partir de $\gamma$ y $\eta$ complejos.
2. Obtener $|H|$ y la potencia promedio a partir de $|E|$ y $\eta$.
3. Calcular el balance de potencia de una onda no polarizada en Brewster.
4. Reconocer qué fórmula de la unidad corresponde a cada pregunta.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "medios_con_perdidas.py", "interfaces_planas.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from medios_con_perdidas import potencia_promedio
from interfaces_planas import coeficientes_fresnel, angulo_brewster
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Trabajar con $\gamma$ y $\eta$ ya dados

En un problema de prueba a veces le entregan directamente $\gamma$ y $\eta$
sin decirle de qué material se trata. Eso alcanza: con esos dos números
complejos puede responder todo.

- La parte real de $\gamma$ dice cuánto se atenúa la onda.
- La parte imaginaria dice cuánta fase acumula.
- El módulo de $\eta$ convierte $|E|$ en $|H|$.
- La fase de $\eta$ dice cuánto se desfasan entre sí, y eso reduce la
  potencia transportada.

### 2.2 Luz no polarizada en una interfaz

La luz del sol o de una ampolleta no tiene una polarización definida: es una
mezcla al azar. Para calcular su reflexión, se toma el promedio de las dos
polarizaciones, cada una con peso $1/2$:

$$
R_{\text{total}} = \tfrac{1}{2}|r_\perp|^{2} + \tfrac{1}{2}|r_\parallel|^{2}.
$$

En Brewster el segundo término se anula, así que la reflexión cae a la mitad
de lo que sería con la perpendicular sola.

## 3. Ecuaciones

**Problema 1 — propagación en un medio con pérdidas:**

$$
\tilde{E}(z) = E_0\,e^{-\gamma z},
\qquad
|H| = \frac{|E|}{|\eta|},
\qquad
S_{\text{prom}} = \frac{1}{2}|E|^{2}\operatorname{Re}\!\left(\frac{1}{\eta^{*}}\right).
$$

La fase del campo en el punto $z$ es $\arg(\tilde{E}) = -\beta z$, medida
respecto de la fase en el origen.

**Problema 2 — Brewster y balance de potencia:**

$$
\tan\theta_B = \frac{n_2}{n_1},
\qquad
\theta_t = \arcsin\!\left(\frac{n_1}{n_2}\sin\theta_B\right),
$$

$$
R_{\text{total}} = \tfrac{1}{2}|r_\perp|^{2} + \tfrac{1}{2}|r_\parallel|^{2},
\qquad
T_{\text{total}} = 1 - R_{\text{total}} .
$$

Un detalle que conviene notar: en Brewster, $\theta_B + \theta_t = 90^\circ$
exactamente. Los rayos reflejado y transmitido salen perpendiculares entre sí.

## 4. Qué significa físicamente

**La atenuación y la fase van por caminos separados.** $\alpha$ solo achica
la amplitud y $\beta$ solo corre la fase. Por eso conviene trabajar con
$\gamma$ completo y separar al final.

**Un $\eta$ complejo reduce la potencia transportada.** Cuando $E$ y $H$ no
están en fase, parte de la energía va y vuelve en vez de avanzar. Eso es lo
que captura la parte real de $1/\eta^{*}$.

**En Brewster la reflexión no se anula, se reduce a la mitad.** Con luz no
polarizada, la componente paralela desaparece pero la perpendicular sigue ahí.
Lo que sí ocurre es que la luz reflejada queda 100 % polarizada.

**Los dos rayos salen en ángulo recto.** Ésa es la manera intuitiva de
recordar Brewster: el dipolo que oscila en el medio 2 no puede radiar en la
dirección en la que oscila, y en Brewster esa dirección coincide justo con la
del rayo reflejado.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: medio con pérdidas ---
gamma = 0.2 + 40.0j     # constante de propagación [1/m]
eta = 120.0 + 20.0j     # impedancia intrínseca [ohm]
campo_inicial = 5.0     # amplitud en z = 0 [V/m]
z_evaluacion = 2.0      # posición donde evaluar [m]

# --- Problema 2: interfaz en Brewster ---
n1 = 1.0   # índice del medio de entrada
n2 = 1.5   # índice del medio de salida

## 6. Implementación

### 6.1 Problema 1 — el campo tras 2 metros

In [ ]:
campo = campo_inicial * np.exp(-gamma * z_evaluacion)

magnitud_E = abs(campo)
fase_E = np.rad2deg(np.angle(campo))
magnitud_H = magnitud_E / abs(eta)
potencia = potencia_promedio(magnitud_E, eta)

### 6.2 Problema 2 — reflexión en el ángulo de Brewster

In [ ]:
theta_B = angulo_brewster(n1, n2)
r_perpendicular, r_paralelo, coseno_t = coeficientes_fresnel(n1, n2, theta_B)
theta_t = np.arccos(coseno_t.real)

R_total = 0.5 * abs(r_perpendicular) ** 2 + 0.5 * abs(r_paralelo) ** 2
T_total = 1.0 - R_total

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Atenuación", "alpha", gamma.real, "1/m"),
        ("Constante de fase", "beta", gamma.imag, "rad/m"),
        ("Campo, parte real", "Re(E)", campo.real, "V/m"),
        ("Campo, parte imaginaria", "Im(E)", campo.imag, "V/m"),
        ("Magnitud del campo", "|E|", magnitud_E, "V/m"),
        ("Fase del campo", "arg(E)", fase_E, "grados"),
        ("Magnitud del campo magnético", "|H|", magnitud_H, "A/m"),
        ("Potencia promedio", "S_prom", potencia, "W/m^2"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Ángulo de Brewster", "theta_B", np.rad2deg(theta_B), "grados"),
        ("Ángulo transmitido", "theta_t", np.rad2deg(theta_t), "grados"),
        ("Suma de ambos", "theta_B + theta_t", np.rad2deg(theta_B + theta_t), "grados"),
        ("Reflexión perpendicular (con signo)", "r_perp", r_perpendicular.real, "-"),
        ("Reflexión paralela (con signo)", "r_par", r_paralelo.real, "-"),
        ("Módulo de la reflexión perpendicular", "|r_perp|", abs(r_perpendicular), "-"),
        ("Módulo de la reflexión paralela", "|r_par|", abs(r_paralelo), "-"),
        ("Potencia reflejada total", "R_total", R_total, "-"),
        ("Potencia transmitida total", "T_total", T_total, "-"),
    ]
)

In [ ]:
print(f"|r_paralelo| = {abs(r_paralelo):.3e}  (debería ser cero en Brewster)")
print(f"theta_B + theta_t = {np.rad2deg(theta_B + theta_t):.6f} grados")
print(f"Atenuación tras {z_evaluacion:.1f} m: "
      f"{magnitud_E / campo_inicial * 100.0:.2f} % del campo inicial")

## 8. Visualización

A la izquierda, cómo se apaga la onda. A la derecha, el mínimo de Fresnel de
la polarización paralela.

In [ ]:
fig, (eje_atenuacion, eje_fresnel) = plt.subplots(1, 2, figsize=(9.5, 4.0))

z = np.linspace(0.0, 8.0, 400)
eje_atenuacion.plot(z, campo_inicial * np.exp(-gamma.real * z))
eje_atenuacion.scatter([z_evaluacion], [magnitud_E], color="black", zorder=5,
                       label="punto evaluado")
eje_atenuacion.set_xlabel("z (m)")
eje_atenuacion.set_ylabel("|E| (V/m)")
eje_atenuacion.set_title("La onda se va apagando")
eje_atenuacion.legend()

angulos = np.deg2rad(np.linspace(0.0, 89.0, 500))
_, rp_curva, _ = coeficientes_fresnel(n1, n2, angulos)
eje_fresnel.plot(np.rad2deg(angulos), np.abs(rp_curva) ** 2, color="tab:orange")
eje_fresnel.axvline(np.rad2deg(theta_B), color="black", linestyle="--",
                    label="Brewster")
eje_fresnel.set_xlabel("Ángulo de incidencia (grados)")
eje_fresnel.set_ylabel("Reflectancia paralela")
eje_fresnel.set_title("La polarización paralela desaparece")
eje_fresnel.legend()

fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**Tras 2 metros queda el 67 % del campo.** Con $\alpha = 0.2$ m$^{-1}$, el
factor es $e^{-0.4} = 0.670$. En potencia la caída es al cuadrado: queda un
45 %.

**La fase acumulada es enorme.** Con $\beta = 40$ rad/m y $z = 2$ m son 80
radianes, o sea unas 12.7 vueltas completas. Por eso el ángulo que reporta la
tabla parece arbitrario: es el resto de dividir por $2\pi$. La fase absoluta
rara vez importa; lo que importa son las diferencias de fase.

**La curva de la derecha toca el fondo justo en la línea punteada.** El valor
de $|r_\parallel|$ es del orden de $10^{-16}$, es decir cero hasta la
precisión de la máquina.

**Los dos ángulos suman 90° exactos.** La tabla lo confirma. Si en una prueba
usted calcula Brewster y esa suma no le da 90°, hay un error en alguna parte.

## 10. Ejercicios para experimentar

            1. Duplique la parte real de `gamma` a `0.4`. ¿Qué fracción del campo queda
               tras 2 m? ¿Y de potencia?
            2. Haga `gamma = 0.0 + 40.0j`. ¿Qué le pasa a la amplitud? ¿Qué tipo de medio
               representa?
            3. Haga `eta = 120.0 + 0.0j`, real puro. ¿Cómo cambia la potencia promedio?
               Explique el rol de la fase de $\eta$.
            4. Cambie `z_evaluacion` a `10.0` m. ¿Cuánto sobrevive?
            5. Ponga `n2 = 1.33` (agua). ¿Dónde queda Brewster? Compare con el ángulo al
               que uno mira un lago de pie en la orilla.
            6. Verifique numéricamente, para varios pares `n1`/`n2`, que
               $\theta_B + \theta_t$ siempre da 90°. ¿Puede demostrarlo a partir de Snell
               y de la definición de Brewster?